#Initialization

In [0]:
%python
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

#Read bronze table

In [0]:
%python
df = spark.table('workspace.bronze.erp_px_cat_g1v2')

#Silver Transformations

##Trimming

In [0]:
%python
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))


## Normalize maintenance flag to boolean

In [0]:
%python
df = df.withColumn(
    "maintenance",
    F.when(F.upper(col("maintenance")) == "YES", F.lit(True))
    .when(F.upper(col("maintenance")) == "NO", F.lit(False))
    .otherwise(None)
)

In [0]:
%python
df.display()

## Renaming columns

In [0]:
%python
RENAME_MAP = {
    "id": "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance_flag"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
       

## Sanity check of dataframe

In [0]:
%python
df.limit(10).display()

# Writing Silver Table

In [0]:
%python
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_product_category")

# Sanity check of Silver Table

In [0]:
SELECT * FROM workspace.silver.erp_product_category LIMIT 10;